In [1]:
import pandas as pd
import numpy as np

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:


import urllib.request
import zipfile
import os

url = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"

urllib.request.urlretrieve(url, "bike_dataset.zip")

with zipfile.ZipFile("bike_dataset.zip", "r") as zip_ref:
    zip_ref.extractall("bike_data")

print("Dataset downloaded successfully!")

Dataset downloaded successfully!


In [3]:


print(os.listdir("bike_data"))

['Readme.txt', 'day.csv', 'hour.csv']


In [4]:


df = pd.read_csv("bike_data/hour.csv")

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [5]:
print(df.head())

   instant      dteday  season  yr  mnth  hr  holiday  weekday  workingday  \
0        1  2011-01-01       1   0     1   0        0        6           0   
1        2  2011-01-01       1   0     1   1        0        6           0   
2        3  2011-01-01       1   0     1   2        0        6           0   
3        4  2011-01-01       1   0     1   3        0        6           0   
4        5  2011-01-01       1   0     1   4        0        6           0   

   weathersit  temp   atemp   hum  windspeed  casual  registered  cnt  
0           1  0.24  0.2879  0.81        0.0       3          13   16  
1           1  0.22  0.2727  0.80        0.0       8          32   40  
2           1  0.22  0.2727  0.80        0.0       5          27   32  
3           1  0.24  0.2879  0.75        0.0       3          10   13  
4           1  0.24  0.2879  0.75        0.0       0           1    1  


In [6]:
print(df.shape)


(17379, 17)


In [7]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 17 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   instant     17379 non-null  int64  
 1   dteday      17379 non-null  object 
 2   season      17379 non-null  int64  
 3   yr          17379 non-null  int64  
 4   mnth        17379 non-null  int64  
 5   hr          17379 non-null  int64  
 6   holiday     17379 non-null  int64  
 7   weekday     17379 non-null  int64  
 8   workingday  17379 non-null  int64  
 9   weathersit  17379 non-null  int64  
 10  temp        17379 non-null  float64
 11  atemp       17379 non-null  float64
 12  hum         17379 non-null  float64
 13  windspeed   17379 non-null  float64
 14  casual      17379 non-null  int64  
 15  registered  17379 non-null  int64  
 16  cnt         17379 non-null  int64  
dtypes: float64(4), int64(12), object(1)
memory usage: 2.3+ MB


In [8]:
print(df.isnull().sum())


instant       0
dteday        0
season        0
yr            0
mnth          0
hr            0
holiday       0
weekday       0
workingday    0
weathersit    0
temp          0
atemp         0
hum           0
windspeed     0
casual        0
registered    0
cnt           0
dtype: int64


In [9]:


print("Total missing values:", df.isnull().sum().sum())

Total missing values: 0


In [10]:
print("Duplicate rows:", df.duplicated().sum())


Duplicate rows: 0


In [11]:
df_clean = df.drop_duplicates().copy()

print("Original dataset:", df.shape)
print("Cleaned dataset:", df_clean.shape)

print("Data cleaning completed!")


Original dataset: (17379, 17)
Cleaned dataset: (17379, 17)
Data cleaning completed!


In [12]:
print("Missing values after cleaning:")
print(df_clean.isnull().sum().sum())

print("Duplicate rows after cleaning:")
print(df_clean.duplicated().sum())


Missing values after cleaning:
0
Duplicate rows after cleaning:
0


In [13]:
threshold = df_clean["cnt"].median()

print("Median rental count:", threshold)

Median rental count: 142.0


In [14]:
df_clean["demand_label"] = np.where(
    df_clean["cnt"] >= threshold,
    "HIGH",
    "LOW"
)

print("Demand labels created successfully!")

Demand labels created successfully!


In [15]:
print(df_clean[["cnt", "demand_label"]].head(10))


   cnt demand_label
0   16          LOW
1   40          LOW
2   32          LOW
3   13          LOW
4    1          LOW
5    1          LOW
6    2          LOW
7    3          LOW
8    8          LOW
9   14          LOW


In [16]:
print(df_clean["demand_label"].value_counts())


demand_label
HIGH    8708
LOW     8671
Name: count, dtype: int64


HIGH and LOW Demand Classification

In this project, bicycle rental demand is divided into two categories: HIGH and LOW.

The median rental count is used as the threshold.

- HIGH: Rental count is greater than or equal to the median.
- LOW: Rental count is less than the median.

This classification converts numerical rental data into two categories for machine learning prediction.

Conclusion: The demand labels are created successfully using Python and NumPy.

In [17]:
# Step 7.1: Prepare the data

# Sort records by date and hour
df_clean = df_clean.sort_values(
    ["dteday", "hr"]
).reset_index(drop=True)

# Use the first 80% for training
split_point = int(len(df_clean) * 0.80)

train_data = df_clean.iloc[:split_point].copy()
test_data = df_clean.iloc[split_point:].copy()

# Calculate threshold using training data only
threshold = train_data["cnt"].median()

# Create demand labels using the training threshold
train_data["demand_label"] = np.where(
    train_data["cnt"] >= threshold, "HIGH", "LOW"
)

test_data["demand_label"] = np.where(
    test_data["cnt"] >= threshold, "HIGH", "LOW"
)

print("Training and testing data prepared!")
print("Training records:", len(train_data))
print("Testing records:", len(test_data))
print("Demand threshold:", threshold)

Training and testing data prepared!
Training records: 13903
Testing records: 3476
Demand threshold: 130.0


In [18]:
# Step 7.2: Select input features

features = [
    "temp",
    "hum",
    "hr",
    "weathersit",
    "season"
]

X_train = train_data[features]
X_test = test_data[features]

y_train = train_data["demand_label"]
y_test = test_data["demand_label"]

print("Input and output selected successfully!")

Input and output selected successfully!


In [19]:
# Step 7.3: Create baseline model

from sklearn.dummy import DummyClassifier

model = DummyClassifier(strategy="most_frequent")

# Train the model
model.fit(X_train, y_train)

print("Baseline model trained successfully!")

Baseline model trained successfully!


In [20]:
# Step 7.4: Make predictions

y_pred = model.predict(X_test)

print("Predictions:")
print(y_pred[:10])

Predictions:
['HIGH' 'HIGH' 'HIGH' 'HIGH' 'HIGH' 'HIGH' 'HIGH' 'HIGH' 'HIGH' 'HIGH']


In [21]:
# Step 7.5: Calculate accuracy

from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Baseline Accuracy:", accuracy)
print("Accuracy Percentage:", accuracy * 100, "%")

Baseline Accuracy: 0.6262945914844649
Accuracy Percentage: 62.629459148446486 %


In [22]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Evaluation libraries imported successfully!")

Evaluation libraries imported successfully!


In [23]:
precision = precision_score(
    y_test, y_pred, pos_label="HIGH", zero_division=0
)

recall = recall_score(
    y_test, y_pred, pos_label="HIGH", zero_division=0
)

f1 = f1_score(
    y_test, y_pred, pos_label="HIGH", zero_division=0
)

print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

Precision: 0.6262945914844649
Recall: 1.0
F1-score: 0.7702105076950292


In [24]:
cm = confusion_matrix(
    y_test, y_pred, labels=["LOW", "HIGH"]
)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[   0 1299]
 [   0 2177]]


In [25]:
print(classification_report(
    y_test,
    y_pred,
    labels=["LOW", "HIGH"],
    zero_division=0
))

              precision    recall  f1-score   support

         LOW       0.00      0.00      0.00      1299
        HIGH       0.63      1.00      0.77      2177

    accuracy                           0.63      3476
   macro avg       0.31      0.50      0.39      3476
weighted avg       0.39      0.63      0.48      3476



In [26]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=["LOW", "HIGH"]))

Confusion Matrix:
[[   0 1299]
 [   0 2177]]


In [27]:
from sklearn.tree import DecisionTreeClassifier

print("Decision Tree imported successfully!")

Decision Tree imported successfully!


In [28]:
dt_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

dt_model.fit(X_train, y_train)

print("Decision Tree model trained successfully!")

Decision Tree model trained successfully!


In [29]:
dt_pred = dt_model.predict(X_test)

print("Predictions:")
print(dt_pred[:10])

Predictions:
['HIGH' 'HIGH' 'HIGH' 'HIGH' 'HIGH' 'HIGH' 'HIGH' 'HIGH' 'HIGH' 'HIGH']


In [30]:
from sklearn.metrics import accuracy_score

dt_accuracy = accuracy_score(y_test, dt_pred)

print("Decision Tree Accuracy:", dt_accuracy)
print("Accuracy Percentage:", dt_accuracy * 100, "%")

Decision Tree Accuracy: 0.8794591484464902
Accuracy Percentage: 87.94591484464902 %


In [31]:
print(classification_report(
    y_test,
    dt_pred,
    labels=["LOW", "HIGH"],
    zero_division=0
))

              precision    recall  f1-score   support

         LOW       0.81      0.89      0.85      1299
        HIGH       0.93      0.87      0.90      2177

    accuracy                           0.88      3476
   macro avg       0.87      0.88      0.87      3476
weighted avg       0.88      0.88      0.88      3476



In [32]:
dt_cm = confusion_matrix(
    y_test,
    dt_pred,
    labels=["LOW", "HIGH"]
)

print("Decision Tree Confusion Matrix:")
print(dt_cm)

Decision Tree Confusion Matrix:
[[1157  142]
 [ 277 1900]]


In [33]:
print("Model Comparison")
print("-------------------------")
print("Dummy Classifier:", accuracy)
print("Decision Tree:", dt_accuracy)

Model Comparison
-------------------------
Dummy Classifier: 0.6262945914844649
Decision Tree: 0.8794591484464902


In [34]:
import joblib

joblib.dump(dt_model, "bike_rental_model.pkl")

print("Model saved successfully!")

Model saved successfully!


In [35]:
from google.colab import files

files.download("bike_rental_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>